# Dataset creation

This is a notebook to take our tsv from uniprot and:
- add mmseqs clusters so we can do train:test splits and try to make sure that proteins with a similar level of homology aren't found in train vs test, which would result in a form of data leakage. Mmseqs clustering was done on a personal laptop because installing it on the mila cluster needed permissions that I didn't want to wade through.
- add prot_param calculations to the base dataframe because they may be useful for correlation-hunting later

### Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import polars as pl
import pandas as pd
import re
import plotly.express as px
from project.utils.strs import PROTEIN_LENGTH_CUTOFF, data_dir, mmseqs_dir
from project.utils.functions import protein_analysis

Read in the dataset - downloaded manually from UniProt

In [ ]:
input_filename = 'uniprotkb_AND_model_organism_9606_2025_09_09'
input_file_end = '.tsv.gz'
df = pl.read_csv(data_dir / (input_filename + input_file_end), separator='\t').rename({'Entry':'id', 'Sequence': 'sequence'})
print(len(df))
reviewed_filter = pl.col('Reviewed') == 'reviewed'
length_filter = pl.col('Length') < PROTEIN_LENGTH_CUTOFF
# Remove proteins with length > length_cutoff, and those not reviewed
df = df.filter(reviewed_filter & length_filter)
print(len(df))

# Looking at the dataset

In [ ]:
df

Get mmseqs clusters, computed at different levels of sequence identity

In [ ]:
for cluster_file in sorted(mmseqs_dir.glob("*cutoff_cluster*")):
    cutoff_val = str(cluster_file).split('_cutoff')[0].split('clusters_')[-1]
    if len(cutoff_val) < 4:
        cutoff_val += '0'
    mmdf = pl.read_csv(cluster_file, separator='\t', has_header=False, new_columns=['cluster_representative', 'cluster_member'])
    cluster_mappings = dict(zip(mmdf['cluster_member'], mmdf['cluster_representative']))
    # Map onto df
    df = df.with_columns(
        pl.col('id').replace_strict(cluster_mappings).alias(f"mmseqs_{cutoff_val}")
    )

Looking at how many unique clusters of proteins we have at different cutoff thresholds:

In [ ]:
unique_counts = {}
for col in df.columns:
    if ('mmseqs' in col):
        unique_counts[col] = len(df[col].unique())
unique_counts

In [ ]:
# Convert dictionary to DataFrame
to_plot_df = pd.DataFrame(list(unique_counts.items()), columns=['Key', 'Value'])

# Extract the numerical threshold from the key (e.g., '0.05' from 'mmseqs_0.05')
to_plot_df['Threshold'] = to_plot_df['Key'].apply(lambda x: float(re.search(r'(\d+\.\d+)', x).group(1)))

# Sort by the numerical threshold to ensure the line plot is ordered correctly
to_plot_df = to_plot_df.sort_values('Threshold')

# Generate the line plot using Plotly Express
fig = px.line(
    to_plot_df,
    x='Threshold',
    y='Value',
    title='MMseqs Clustering Size vs. min-seq-id Threshold',
    markers=True, # Display markers for individual data points
    labels={'Threshold': 'MMseqs Clustering Threshold', 'Value': 'Number of Clusters'}
)

# Customize the plot appearance
fig.update_traces(line=dict(width=3))

# Display the figure (if running in an interactive environment like a Jupyter notebook)
# fig.show()

With higher threshold, we're saying that we only want proteins that are very similar to be clustered together, which is why we get many more clusters. With lower threshold, we're permitting proteins to be grouped together that may be more distantly related. 

Adding prot_param calculated properties

First, see how many proteins have ambiguous amino acids.

In [ ]:
# Define the ambiguity characters
ambiguity_pattern = r"[UOBZJX]"

# Add columns for tracking
df_with_stats = df.with_columns([
    # Total number of ambiguity characters in each sequence
    pl.col("sequence").str.count_matches(ambiguity_pattern).alias("ambiguity_count")
])

# 3. Calculate your summary statistics
stats = df_with_stats.select([
    # How many proteins had at least one ambiguity
    pl.col("ambiguity_count").filter(pl.col("ambiguity_count") > 0).count().alias("proteins_with_ambiguities"),
    
    # Total sum of all ambiguities across the dataset
    pl.col("ambiguity_count").sum().alias("total_ambiguities")
])

print(stats)

This isn't bad. If we modify or overlook ambuguities we should only have a minor effect. We can use our protein analysis function to process the sequences.

In [ ]:
# The list of standard amino acids
amino_acids = ["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y"]
# Create a list of fields for the amino acid counts
amino_acid_fields = [pl.Field(aa, pl.Float64) for aa in amino_acids]
# Have to specify the schema for the struct column we are creating
struct_schema = [
    pl.Field("amino_acid_percent", pl.Struct(amino_acid_fields)),
    pl.Field("flexibility", pl.List(pl.Float64)),
    pl.Field("instability_index", pl.Float64),
    pl.Field("gravy", pl.Float64),
    pl.Field("secondary_structure_fraction", pl.List(pl.Float64)),
    pl.Field("isoelectric_point", pl.Float64),
    pl.Field("charge_at_ph4_7", pl.Float64),
    pl.Field("charge_at_ph7_2", pl.Float64),
    pl.Field("charge_at_ph8", pl.Float64)
]
# Dataframe with 'fundamental characteristics' (easily calculable) that mostly depend on aggregate contributions of amino acids
df = df.with_columns(pl.col("sequence").map_batches(lambda x: pl.Series(protein_analysis(x), dtype=pl.Struct(struct_schema)), return_dtype=pl.Struct(struct_schema)).alias("Properties")).unnest("Properties")

In [ ]:
df

In [ ]:
df.write_parquet(data_dir / (input_filename + '_annotated' + '.parquet.gz'))